# KJSS-2026-0066 · Phase 1A: 2025 MLB 경기 표본 감사

**목적:** 2025년 3~10월 MLB 일정의 `gameType`, 종료 상태, 중복 `gamePk`를 확인합니다. 기존 학습이나 모델 가중치는 변경하지 않습니다.

두 코드 셀을 순서대로 실행하고, 내려받은 `mlb_2025_sample_audit.zip`을 채팅에 업로드해 주세요.

In [ ]:
!pip -q install mlb-statsapi pandas

In [ ]:
import statsapi, pandas as pd, json, time, zipfile
from collections import Counter
from pathlib import Path
from google.colab import files

SEASON = 2025
months = [(f'{SEASON}-{m:02d}-01', (f'{SEASON+1}-01-01' if m == 12 else f'{SEASON}-{m+1:02d}-01')) for m in range(3, 11)]

# 3~10월 각각 첫날부터 말일까지 조회합니다.
raw_rows = []
old_rows = []
for start, end_exclusive in months:
    end = (pd.Timestamp(end_exclusive) - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    # 원래 노트북의 statsapi.schedule 방식 재현
    flat = statsapi.schedule(start_date=start, end_date=end, sportId=1)
    for g in flat:
        old_rows.append({
            'gamePk': str(g.get('game_id', '')),
            'game_date': g.get('game_date', ''),
            'flattened_status': g.get('status', ''),
        })
    # 원본 일정 응답에서 gameType 직접 확인
    response = statsapi.get('schedule', {'sportId': 1, 'startDate': start, 'endDate': end})
    for day in response.get('dates', []):
        for game in day.get('games', []):
            status = game.get('status', {})
            raw_rows.append({
                'gamePk': str(game.get('gamePk', '')),
                'date': str(day.get('date', '')),
                'gameType': str(game.get('gameType', '')),
                'abstractGameState': str(status.get('abstractGameState', '')),
                'detailedState': str(status.get('detailedState', '')),
                'codedGameState': str(status.get('codedGameState', '')),
                'home_id': game.get('teams', {}).get('home', {}).get('team', {}).get('id'),
                'away_id': game.get('teams', {}).get('away', {}).get('team', {}).get('id'),
            })
    print(f'{start[:7]}  원래 방식 {len(flat):4}건 | 원본 일정 {sum(r["date"].startswith(start[:7]) for r in raw_rows):4}건')
    time.sleep(0.3)

raw = pd.DataFrame(raw_rows)
old = pd.DataFrame(old_rows)
old_final = old[old['flattened_status'].eq('Final')].copy()
regular_final = raw[(raw['gameType'] == 'R') & (raw['abstractGameState'] == 'Final')].copy()
regular_unique = regular_final.drop_duplicates('gamePk').copy()
old_game_ids = set(old_final['gamePk'])
reg_game_ids = set(regular_unique['gamePk'])

print('\n========== 감사 결과 ==========')
print('원래 방식 Final 총 행:', len(old_final))
print('원래 방식 Final 고유 gamePk:', old_final['gamePk'].nunique())
print('정규시즌(R) Final 총 행:', len(regular_final))
print('정규시즌(R) Final 고유 gamePk:', len(reg_game_ids))
print('기존 Final 중 정규시즌 이외/확인불가 gamePk:', len(old_game_ids - reg_game_ids))
print('정규시즌 고유 gamePk 중 기존 Final에 없는 것:', len(reg_game_ids - old_game_ids))
print('원본 gameType별 행 수:')
print(raw.groupby(['gameType', 'abstractGameState']).size().sort_values(ascending=False).to_string())

out = Path('/content/mlb_audit')
out.mkdir(exist_ok=True)
raw.to_csv(out/'all_schedule_games.csv', index=False, encoding='utf-8-sig')
old_final.to_csv(out/'original_final_games.csv', index=False, encoding='utf-8-sig')
regular_unique.to_csv(out/'regular_season_games.csv', index=False, encoding='utf-8-sig')
summary = {
    'season': SEASON,
    'old_final_rows': int(len(old_final)),
    'old_final_unique_gamePk': int(old_final['gamePk'].nunique()),
    'regular_final_rows': int(len(regular_final)),
    'regular_final_unique_gamePk': int(len(reg_game_ids)),
    'old_minus_regular_unique_ids': int(len(old_game_ids - reg_game_ids)),
    'regular_minus_old_unique_ids': int(len(reg_game_ids - old_game_ids)),
    'game_type_counts': {str(k): int(v) for k, v in raw.groupby('gameType').size().items()},
    'duplicated_raw_gamePk_rows': int(raw.duplicated('gamePk').sum()),
}
(out/'audit_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
zip_path = Path('/content/mlb_2025_sample_audit.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(out.iterdir()):
        z.write(p, arcname=p.name)
print('\n저장:', zip_path)
files.download(str(zip_path))